# 02 — Predicting outage severity (Objective 1)

**Target:** peak % of customers out per (storm episode, county), as `log(1 + percent)` for regression and low (<1%) / medium (1–10%) / high (≥10%) for classification.
**Split:** train 2014–2019, validate 2020 (tuning / early stopping), test 2021–2022.
**Models:** storm-type baseline → linear / logistic → random forest → LightGBM. **SHAP** explains the LightGBM regressor.

In [ ]:
# Works on Colab (Drive) or locally (repo checkout)
import os, sys, pathlib, warnings
warnings.filterwarnings("ignore")
try:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT = pathlib.Path("/content/drive/MyDrive/LightsOut")
    ZIP = pathlib.Path("/content/drive/MyDrive/LightsOut_code.zip")
    if ZIP.exists():  # refresh code (src/, notebooks/) from the latest uploaded zip; data/ is untouched
        import zipfile; zipfile.ZipFile(ZIP).extractall(PROJECT)
    !pip -q install polars lifelines scikit-survival shap geopandas lightgbm
except ImportError:
    PROJECT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(PROJECT / "src"))
os.chdir(PROJECT)

import numpy as np, pandas as pd, matplotlib.pyplot as plt
from lightsout import config as C, viz
viz.setup()
df = pd.read_parquet(C.PROCESSED / "model_table.parquet")
print(f"{len(df):,} (storm episode, county) units, {df.fips.nunique():,} counties, years {df.year.min()}-{df.year.max()}")

In [ ]:
from lightsout import models_severity as S
print(df.groupby("split")["severity"].value_counts(normalize=True).unstack().round(3))
fit = S.fit_all(df)

In [ ]:
res_val, res_test = S.results_table(fit, "val"), S.results_table(fit, "test")
res_test.to_csv(C.REPORTS / "severity_results_test.csv")
(C.REPORTS / "severity_results_test.md").write_text(res_test.to_markdown())
print("Validation (2020)"); display(res_val)
print("Test (2021-2022)"); display(res_test)

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay
te = fit["truth"]["test"]
fig, axes = plt.subplots(1, 2, figsize=(12, 4.6))
ConfusionMatrixDisplay.from_predictions(te.severity.astype(str), fit["pred"]["test"]["LightGBM|clf"],
    labels=["low", "medium", "high"], normalize="true", cmap=viz.SEQUENTIAL_CMAP, ax=axes[0],
    values_format=".2f", colorbar=False)
axes[0].set_title("LightGBM severity class (row-normalized, test)"); axes[0].grid(False)
p = fit["pred"]["test"]["LightGBM|reg"]
axes[1].scatter(te.log_peak_pct, p, s=6, alpha=0.25, color=viz.CATEGORICAL[0], linewidths=0)
lim = [0, max(te.log_peak_pct.max(), p.max())]
axes[1].plot(lim, lim, color=viz.MUTED, linewidth=1, linestyle="--")
axes[1].set_xlabel("Actual log(1 + % out)"); axes[1].set_ylabel("Predicted")
axes[1].set_title("LightGBM regression: predicted vs actual (test)")
viz.save(fig, "06_severity_test_diagnostics"); fig

In [ ]:
# per storm type: does the best model beat the baseline everywhere?
pt = fit["pred"]["test"]
by_type = pd.DataFrame({
    "n": te.groupby("storm_type").size(),
    "baseline MAE": (te.log_peak_pct - pt["Baseline (storm type)|reg"]).abs().groupby(te.storm_type).mean(),
    "LightGBM MAE": (te.log_peak_pct - pt["LightGBM|reg"]).abs().groupby(te.storm_type).mean(),
}).round(3).sort_values("n", ascending=False)
by_type

## SHAP: what drives predicted severity?

In [ ]:
import shap
sv, X = S.shap_values(fit, n=5000)
plt.figure()
shap.plots.beeswarm(sv, max_display=15, show=False, color=viz.SEQUENTIAL_CMAP)
plt.title("SHAP values, LightGBM severity regressor (test sample)")
fig = plt.gcf(); viz.save(fig, "07_shap_summary"); fig

In [ ]:
imp = pd.Series(np.abs(sv.values).mean(0), index=X.columns).sort_values(ascending=False)
imp.head(15).round(4).to_frame("mean |SHAP|")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
for ax, feat in zip(axes, ["max_wind_kts", "county_hist_log_peak"]):
    j = list(X.columns).index(feat)
    ax.scatter(X[feat], sv.values[:, j], s=6, alpha=0.3, color=viz.CATEGORICAL[0], linewidths=0)
    ax.axhline(0, color=viz.AXIS, linewidth=1)
    ax.set_xlabel(feat); ax.set_ylabel("SHAP value"); ax.set_title(f"SHAP dependence: {feat}")
viz.save(fig, "08_shap_dependence"); fig